# 03 Gamma Forecast Impact

This notebook quantifies how the selected Gamma RPF case can inflate 7-day-ahead forecast error. The smoke run always writes the data-error-only benchmark; full forecast mode adds model forecasts.


## 1. Imports And Paths

Load config and confirm the final Gamma dataset path.


In [ ]:
from pathlib import Path
import sys

# Keep notebook imports stable whether the notebook is run from JupyterLab,
# VS Code, or the repository root.
article_root = Path.cwd()
while article_root.name != "2_journal_article":
    if article_root.parent == article_root:
        raise RuntimeError("Could not locate publication/2_journal_article")
    article_root = article_root.parent
notebook_dir = article_root / "notebooks"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

import _experiment_helpers as h

cfg = h.load_config(article_root)
paths = h.article_paths(article_root, cfg)
h.ensure_output_dirs(paths)
print(f"Article root: {article_root}")
print(f"Config schema: {cfg['schema_version']}")
print(f"Output root: {paths.outputs}")

print(article_root / cfg["paths"]["gamma_dataset_path"])
print(f"Full forecast: {cfg['execution']['run_full_forecast']}")


## 2. Load Gamma And Confirm Scope

Gamma must contain exactly one site over the one-year Beta window.


In [ ]:
gamma = h.load_dataset(article_root, cfg, "gamma")
site = gamma["substation_id"].iloc[0]
print(f"Gamma site: {site}")
h.dataset_summary(gamma, "Gamma")


## 3. Run Forecast-Impact Workflow

The workflow writes the Gamma series, data-error benchmark, metrics, curated tables, figures, and manifest. Full mode also creates rolling forecast examples and model forecasts.


In [ ]:
result = h.run_gamma_forecast_impact(article_root)
print(result["status"], result["gamma_site"])
result["metrics"]


## 4. Interpret The Data-Error-Only Benchmark

The `data_error_only` row answers a narrow but powerful question: if a forecast perfectly predicted the raw data, how wrong would it be against the reference series because of the RPF sign issue alone?


In [ ]:
result["metrics"].sort_values(["data_condition", "model"])
